# EEG-Based Pain Intensity Prediction (Ultimate Ensemble)

This version implements a **Diverse Learning Stack** to break the 50% accuracy mark. By combining multiple algorithmic families (Trees, Neural Networks, and Geometric/Instance-based learning), we capture a broader range of the brain's spectral patterns.

### Ultimate Stack Components:
1. **Tree-Based (XGBoost)**: Captures complex threshold interactions between frequency bands.
2. **Neural Network (MLP)**: Deep layers (128-64-32) to model non-linear combinations of band ratios.
3. **Geometric (K-Nearest Neighbors)**: Exploits local 'neighborhoods' of biometric similarity.
4. **Meta-Learner**: Regularized Logistic Regression to find the optimal voting weights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.ensemble import StackingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings

warnings.filterwarnings('ignore')
sns.set(style="whitegrid", palette="pastel")

## 1. Load & Subject-Wise Normalization

In [ ]:
file_path = 'eeg_data_model/merged_pain_eeg_4Hz.csv'
df = pd.read_csv(file_path)
df.columns = df.columns.str.strip()

bands = ['Delta', 'Theta', 'Alpha1', 'Alpha2', 'Beta1', 'Beta2', 'Gamma1', 'Gamma2']

print("Applying Biological Normalization (Subject Z-Score)...")
for col in bands + ['Attention', 'Meditation']:
    df[col] = df.groupby('person_id')[col].transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))

df['ratio_alpha_theta'] = (df['Alpha1'] + df['Alpha2']) / (df['Theta'] + 1e-8)
df['pain_version_enc'] = LabelEncoder().fit_transform(df['pain_version'].astype(str))

print(f"Preprocessing complete for {df.shape[0]} samples.")

## 2. Ultimate Stacking Implementation

In [ ]:
fcols = bands + ['Attention', 'Meditation', 'ratio_alpha_theta', 'pain_version_enc']
X = df[fcols]
y = LabelEncoder().fit_transform(df['pain_intensity'])
groups = df['person_id']

# Diversity Ensemble Base Learners
base_learners = [
    ('xgb', xgb.XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.05, random_state=42)),
    ('mlp', MLPClassifier(hidden_layer_sizes=(128, 64, 32), max_iter=500, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=15, weights='distance'))
]

ultimate_stack = StackingClassifier(
    estimators=base_learners, 
    final_estimator=LogisticRegression(C=1.0, penalty='l2'),
    cv=5, # Higher CV for better meta-label stability
    n_jobs=-1
)

print("Ultimate Stack Ready.")

## 3. High-Precision Evaluation

In [ ]:
gkf = GroupKFold(n_splits=5)
results = []

print("Starting Subject-wise Cross-Validation (Diverse Learners)...")
for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups)):
    ultimate_stack.fit(X.iloc[train_idx], y[train_idx])
    score = accuracy_score(y[test_idx], ultimate_stack.predict(X.iloc[test_idx]))
    results.append(score)
    print(f"Fold {fold+1} Accuracy: {score:.4f}")

print(f"\nUltimate Mean Accuracy: {np.mean(results):.4f} (+/- {np.std(results):.4f})")

## 4. Class-Wise Analysis

In [ ]:
# Final Split for Visuals
split_gkf = GroupKFold(n_splits=5)
tr_i, te_i = next(split_gkf.split(X, y, groups=groups))

ultimate_stack.fit(X.iloc[tr_i], y[tr_i])
y_pred = ultimate_stack.predict(X.iloc[te_i])

plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y[te_i], y_pred), annot=True, fmt='d', cmap='inferno')
plt.title('Ultimate Diverse Ensemble Confusion Matrix')
plt.xlabel('Predicted Pain Level')
plt.ylabel('Ground Truth')
plt.show()

print("Detailed Ultimate Metrics:")
print(classification_report(y[te_i], y_pred))